In [31]:
import pandas as pd
import re
import nltk

NLTK_AVAILABLE = {}
for resource, resource_path in {
    "punkt": "tokenizers/punkt",
    "averaged_perceptron_tagger": "taggers/averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng": "taggers/averaged_perceptron_tagger_eng",
    "wordnet": "corpora/wordnet",
    "stopwords": "corpora/stopwords",
}.items():
    try:
        nltk.data.find(resource_path)
        NLTK_AVAILABLE[resource] = True
    except LookupError:
        NLTK_AVAILABLE[resource] = False

print("NLTK resource status:", NLTK_AVAILABLE)

NLTK resource status: {'punkt': True, 'averaged_perceptron_tagger': True, 'averaged_perceptron_tagger_eng': False, 'wordnet': False, 'stopwords': True}


In [32]:
pain_points = '../output/challenges.csv'
expectations = '../output/expectations.csv'

df_pain_points = pd.read_csv(pain_points)
df_expectations = pd.read_csv(expectations)

In [33]:
df_pain_points.shape, df_expectations.shape

((424, 2), (433, 2))

In [34]:
# remove records with less than or equal to 4 words
df_pain_points = df_pain_points[df_pain_points['pain_points'].str.split().str.len() > 4]
df_expectations = df_expectations[df_expectations['expectations'].str.split().str.len() > 4]

In [35]:
df_pain_points.shape, df_expectations.shape

((324, 2), (309, 2))

In [36]:
df_pain_points.sample(5)

,focus_group,pain_points
42,BAA BAA Supervisors,Inconsistent Note Taking: Variations in how st...
256,DOCE Supervisors,Volume Challenges: IPS becomes cumbersome as c...
238,DOCE Admin Aide,Staff often support requests intended for othe...
11,CPO Central Permit Office,No Automatic Cross-System Notifications: Permi...
59,CPO CPO Co-Ordinator,Property Research Complexity: Multiple searche...


In [37]:
from nltk import pos_tag, word_tokenize

def safe_word_tokenize(text):
    s = str(text)
    try:
        return word_tokenize(s)
    except LookupError:
        return re.findall(r"[A-Za-z]+", s)

def safe_pos_tag(tokens):
    try:
        return pos_tag(tokens)
    except LookupError:
        return [(tok, "NN") for tok in tokens]

def extract_keywords(text):
    tokens = safe_word_tokenize(text)
    tagged = safe_pos_tag(tokens)
    keywords = [
        word.lower()
        for word, tag in tagged
        if tag.startswith("NN") or tag.startswith("JJ")
    ]
    return keywords

In [52]:
from nltk import RegexpParser

grammar = r"""
NP:
    {<JJ>*<NN.*>+}
"""

chunker = RegexpParser(grammar)

df_pain_points["processed_text"] = (
    df_pain_points["pain_points"]
    .fillna("")
    .astype(str)
    .str.replace(r"[^A-Za-z\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.lower()
)

df_expectations["processed_text"] = (
    df_expectations["expectations"]
    .fillna("")
    .astype(str)
    .str.replace(r"[^A-Za-z\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.lower()
)

def extract_phrases(text):
    tokens = safe_word_tokenize(text)
    tagged = safe_pos_tag(tokens)
    tree = chunker.parse(tagged)

    phrases = []
    for subtree in tree.subtrees():
        if subtree.label() == "NP":
            phrase = " ".join(
                word.lower()
                for word, _ in subtree.leaves()
            )
            phrases.append(phrase)

    return phrases

df_pain_points[["pain_points", "processed_text"]].head(3)

,pain_points,processed_text
3,Split Permit & Code Systems: Permit and Code E...,split permit code systems permit and code enfo...
4,Scattered Information: Property and permit inf...,scattered information property and permit info...
5,Research Complexity: Users must search multipl...,research complexity users must search multiple...


In [39]:
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures

tokens = []
for text in df_pain_points["processed_text"]:
    tokens.extend(safe_word_tokenize(text))

finder = BigramCollocationFinder.from_words(tokens)
finder.apply_freq_filter(3)

bigrams = finder.score_ngrams(
    BigramAssocMeasures().pmi
)

for bg, score in bigrams[:50]:
    print(bg, score)

('compliance', 'engine') 9.915879378835774
('making', 'it') 9.330916878114618
('department', 'dependencies') 9.10852445677817
('follow', 'up') 9.10852445677817
('single', 'source') 8.915879378835774
('paper', 'files') 8.693486957499326
('cross', 'department') 8.523561956057012
('city', 'departments') 8.215439660694681
('code', 'enforcement') 8.093877680813769
('too', 'many') 8.008988783227256
('time', 'consuming') 7.915879378835774
('a', 'single') 7.745954377393462
('switch', 'between') 7.593951283948411
('time', 'responding') 7.50084187955693
('zoning', 'reviews') 7.482919971559667
('heavily', 'on') 7.4564477601984755
('spend', 'significant') 7.413379038306591
('cannot', 'easily') 7.3715588626119635
('data', 'entry') 7.371558862611963
('of', 'truth') 7.330916878114618
('source', 'of') 7.330916878114618
('requires', 'extensive') 7.252914366113345
('supporting', 'documentation') 7.201633861169651
('other', 'tools') 7.108524456778169
('the', 'same') 7.108524456778169
('depends', 'on') 7.

In [40]:
from nltk.collocations import TrigramCollocationFinder
from nltk.metrics import TrigramAssocMeasures

finder = TrigramCollocationFinder.from_words(tokens)

finder.apply_freq_filter(3)

trigrams = finder.score_ngrams(
    TrigramAssocMeasures().pmi
)

In [53]:
from collections import Counter

phrase_counter = Counter()

for text in df_pain_points["processed_text"]:

    phrases = extract_phrases(text)

    phrase_counter.update(phrases)

raw_categories = (
    phrase_counter
    .most_common(100)
)

print(raw_categories)

[('split permit code systems permit and code enforcement information exists in separate systems and workflows', 1), ('scattered information property and permit information is distributed across ips camino as onbase and historical archives', 1), ('research complexity users must search multiple systems to understand complete property history', 1), ('difficult information retrieval finding relevant information often feels like hunting through multiple systems', 1), ('limited ips camino integration information does not flow seamlessly between systems', 1), ('no shared permit violation visibility permit reviewers cannot easily see related code information', 1), ('duplicate research effort staff repeatedly search for the same information across systems', 1), ('no automatic cross system notifications permit and code workflows do not automatically notify each other', 1), ('limited status transparency applicants cannot easily see permit status and review progress', 1), ('review status gaps diff

In [ ]:
# Sentiment analysis for pain points based on processed_text (VADER)
from nltk.sentiment import SentimentIntensityAnalyzer

def get_vader_analyzer():
    try:
        return SentimentIntensityAnalyzer()
    except LookupError:
        try:
            nltk.download("vader_lexicon", quiet=True)
            return SentimentIntensityAnalyzer()
        except Exception:
            return None

sia = get_vader_analyzer()

def classify_vader_sentiment(text: str):
    t = str(text).strip()
    if not t:
        return "neutral"
    if sia is None:
        return "neutral"

    compound = sia.polarity_scores(t)["compound"]
    if compound >= 0.05:
        return "positive"
    if compound <= -0.05:
        return "negative"
    return "neutral"

df_pain_points["sentiment"] = df_pain_points["processed_text"].apply(classify_vader_sentiment)
df_pain_points[["focus_group", "processed_text", "sentiment"]].sample(5, random_state=42)

,focus_group,pain_points,sentiment
171,NBD NBD Internal,Contact Management Issues: The IPS contact/add...,negative
143,DOCE CommercialPermitElectrical Inspectors,Lack of Coordination: Electrical inspectors id...,negative
176,NBD NBD Internal,Limited Data Sharing: Information is not easil...,negative
14,CPO Central Permit Office,Review Status Gaps: Difficult for applicants t...,negative
234,DOCE Admin Aide,Staff lose significant time handling repeat ca...,neutral


In [43]:
# Quick check for CPO rows mentioned
df_pain_points.loc[
    df_pain_points["focus_group"].eq("CPO Central Permit Office"),
    ["focus_group", "pain_points", "sentiment"]
] .head(10)

,focus_group,pain_points,sentiment
3,CPO Central Permit Office,Split Permit & Code Systems: Permit and Code E...,negative
4,CPO Central Permit Office,Scattered Information: Property and permit inf...,negative
5,CPO Central Permit Office,Research Complexity: Users must search multipl...,negative
6,CPO Central Permit Office,Difficult Information Retrieval: Finding relev...,negative
8,CPO Central Permit Office,Limited IPS-Camino Integration: Information do...,negative
9,CPO Central Permit Office,No Shared Permit/Violation Visibility: Permit ...,negative
10,CPO Central Permit Office,Duplicate Research Effort: Staff repeatedly se...,negative
11,CPO Central Permit Office,No Automatic Cross-System Notifications: Permi...,negative
13,CPO Central Permit Office,Limited Status Transparency: Applicants cannot...,negative
14,CPO Central Permit Office,Review Status Gaps: Difficult for applicants t...,negative


In [44]:
df_pain_points.loc[df_pain_points['sentiment'] == 'negative'].shape, df_pain_points.loc[df_pain_points['sentiment'] == 'positive'].shape, df_pain_points.loc[df_pain_points['sentiment'] == 'neutral'].shape

((235, 4), (2, 4), (87, 4))

In [45]:
# total pain points per focus group (show all groups)
pain_points_summary = df_pain_points.groupby("focus_group", dropna=False).agg(
    total_pain_points=pd.NamedAgg(column="pain_points", aggfunc="count"),
    negative_pain_points=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "negative").sum()),
    positive_pain_points=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "positive").sum()),
    neutral_pain_points=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "neutral").sum())
).reset_index()

# Validation: these two numbers should match
print("distinct focus groups in data:", df_pain_points["focus_group"].nunique(dropna=False))
print("rows in summary:", len(pain_points_summary))

pain_points_summary.sort_values(by=["total_pain_points", "focus_group"], ascending=[False, True])

distinct focus groups in data: 16
rows in summary: 16


,focus_group,total_pain_points,negative_pain_points,positive_pain_points,neutral_pain_points
6,DOCE Admin Aide,34,17,0,17
8,DOCE CommercialPermitElectrical Inspectors,30,26,0,4
10,DOCE Housing Inspectors,30,22,1,7
7,DOCE Building Inspectors,29,20,0,9
11,DOCE Office Manager,28,17,1,10
12,DOCE Supervisors,26,19,0,7
9,DOCE Fire Prevention Bureau,25,18,0,7
4,CPO CPO Co-Ordinator,22,18,0,4
5,CPO Central Permit Office,21,15,0,6
15,NBD NBD Internal,14,13,0,1


In [46]:
# show only the pain points that are positive
top_focus_groups = pain_points_summary.sort_values(by="total_pain_points", ascending=False).head(10)["focus_group"].tolist()
df_pain_points.loc[
    df_pain_points["focus_group"].isin(top_focus_groups)
    & df_pain_points["sentiment"].eq("positive"),
    ["focus_group", "pain_points", "sentiment"]
]

,focus_group,pain_points,sentiment
304,DOCE Office Manager,Searching across properties and cases is ineff...,positive
422,DOCE Housing Inspectors,Large Dropdown Lists: Long dropdown menus make...,positive


In [47]:
#convert positive sentiment to neutral for pain points
df_pain_points.loc[df_pain_points['sentiment'] == 'positive', 'sentiment'] = 'neutral'

In [ ]:
# Sentiment analysis for expectations based on processed_text (VADER)
if "classify_vader_sentiment" not in globals():
    from nltk.sentiment import SentimentIntensityAnalyzer

    def get_vader_analyzer():
        try:
            return SentimentIntensityAnalyzer()
        except LookupError:
            try:
                nltk.download("vader_lexicon", quiet=True)
                return SentimentIntensityAnalyzer()
            except Exception:
                return None

    sia = get_vader_analyzer()

    def classify_vader_sentiment(text: str):
        t = str(text).strip()
        if not t:
            return "neutral"
        if sia is None:
            return "neutral"

        compound = sia.polarity_scores(t)["compound"]
        if compound >= 0.05:
            return "positive"
        if compound <= -0.05:
            return "negative"
        return "neutral"

df_expectations["sentiment"] = df_expectations["processed_text"].apply(classify_vader_sentiment)

# total expectations per focus group (show all groups)
expectations_summary = df_expectations.groupby("focus_group", dropna=False).agg(
    total_expectations=pd.NamedAgg(column="expectations", aggfunc="count"),
    negative_expectations=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "negative").sum()),
    positive_expectations=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "positive").sum()),
    neutral_expectations=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "neutral").sum())
).reset_index()

# Validation: these two numbers should match
print("distinct focus groups in data:", df_expectations["focus_group"].nunique(dropna=False))
print("rows in summary:", len(expectations_summary))

expectations_summary.sort_values(by=["total_expectations", "focus_group"], ascending=[False, True])

distinct focus groups in data: 16
rows in summary: 16


,focus_group,total_expectations,negative_expectations,positive_expectations,neutral_expectations
10,DOCE Housing Inspectors,31,2,15,14
12,DOCE Supervisors,30,1,14,15
7,DOCE Building Inspectors,27,1,17,9
5,CPO Central Permit Office,26,0,19,7
6,DOCE Admin Aide,26,1,17,8
8,DOCE CommercialPermitElectrical Inspectors,26,1,16,9
9,DOCE Fire Prevention Bureau,26,0,18,8
4,CPO CPO Co-Ordinator,25,0,17,8
11,DOCE Office Manager,21,1,15,5
3,CPC CPC,13,0,11,2


In [49]:
# display expectations that are negative
neg_expectations = df_expectations.loc[
    df_expectations["sentiment"] == "negative",
    ["focus_group", "expectations", "sentiment"]
]
neg_expectations.sample(n=min(10, len(neg_expectations)), random_state=42)

,focus_group,expectations,sentiment
88,DOCE Building Inspectors,Consolidated Ownership Records: Eliminate dupl...,negative
141,DOCE CommercialPermitElectrical Inspectors,Unified Scheduling Platform: Eliminate duplica...,negative
397,DOCE Housing Inspectors,Missing Inspection Tracking: Report on cases w...,negative
215,DOCE Admin Aide,Eliminate duplicate data entry where possible.,negative
313,DOCE Office Manager,Elimination of duplicate data entry.,negative
234,DOCE Supervisors,Missed Case Identification: Automatically iden...,negative
427,DOCE Housing Inspectors,Lead Risk Identification: Automatically identi...,negative


In [50]:
#convert negative sentiment to neutral for expectations
df_expectations.loc[df_expectations['sentiment'] == 'negative', 'sentiment'] = 'neutral'

In [51]:
#save as csv
df_pain_points.to_csv("../output/clean_challenges.csv", index=False)
df_expectations.to_csv("../output/clean_expectations.csv", index=False)